In [ ]:
import sys

dbutils.widgets.text('project_root', '', 'Workspace path containing agoda_etl')  # noqa: F821 - provided by Databricks
dbutils.widgets.text('manifest_path', '', 'Manifest path')  # noqa: F821 - provided by Databricks
project_root = dbutils.widgets.get('project_root').strip()  # noqa: F821 - provided by Databricks
manifest_path = dbutils.widgets.get('manifest_path').strip()  # noqa: F821 - provided by Databricks

if not project_root:
    raise ValueError('Set project_root to the Workspace folder containing agoda_etl')
if not manifest_path:
    raise ValueError('Set manifest_path to a run_manifest.json file in the UC Volume')

sys.path.insert(0, project_root)
from agoda_etl import config  # noqa: E402 - project_root is added first
from agoda_etl.bronze import run_bronze_ingestion  # noqa: E402 - project_root is added first


In [ ]:
result = run_bronze_ingestion(spark, manifest_path)  # noqa: F821 - provided by Databricks
if result['status'] != 'success':
    raise RuntimeError(result)

print(result)
display(spark.table(config.BRONZE_TABLE).where(  # noqa: F821 - provided by Databricks
    "batch_id = '{}'".format(result['batch_id'].replace("'", "''"))
).limit(20))
